# VR Log Export -- Position (cm) + Deduplication

Prepares a raw `VRlog_*.txt` for sharing:

1. Adds a `Location_cm` column next to the raw `Location_au` position value. The conversion
   factor is derived empirically from this log's own observed AU range, capped at the known
   393-AU track length (same cap `BehavioralDataFiltering.process_data_with_speed_filtering`
   uses), scaled onto the ~130.03 cm physical lap length computed from the wheel-calibration
   constants (`single_revolution_VR`, `single_revolution_treadmill`, `single_lap_VR`) in the
   `# 3. Spatial discretization` block of `Preprocess.py`.
2. Removes duplicate rows: if two `p` (position) rows share the same `ElapsedTime(seconds)`
   value, only the first is kept. Other event types (`s`, `n`, `z`, `r`, `t`, `f`, `sequence`)
   are never dropped by this step, even when they share a timestamp with each other or with a
   position row -- e.g. `z` (RZ hit) and `r` (reward) commonly fire at the exact same instant.

Nothing here modifies `Preprocess.py` or any existing script; this is a standalone export.

In [44]:
import os
import numpy as np
import pandas as pd

TAB = chr(9)

## 1. Point this at a VR log file

In [45]:
VRLOG_PATH = r"D:\V1_SpatialModulation\2p\V1_prism\JSY054_ChronicImaging\251031_JSY_JSY054_SpMod_Day2\TSeries-10312025-1751-001\VRlog_JSY038_10312025_06-08-26_forSharing.txt"   # <-- edit per session

assert os.path.isfile(VRLOG_PATH), f"File not found: {VRLOG_PATH}"
OUTPUT_PATH = os.path.splitext(VRLOG_PATH)[0] + "_forSharing.txt"
print(f"Input:  {VRLOG_PATH}")
print(f"Output: {OUTPUT_PATH}")

Input:  D:\V1_SpatialModulation\2p\V1_prism\JSY054_ChronicImaging\251031_JSY_JSY054_SpMod_Day2\TSeries-10312025-1751-001\VRlog_JSY038_10312025_06-08-26_forSharing.txt
Output: D:\V1_SpatialModulation\2p\V1_prism\JSY054_ChronicImaging\251031_JSY_JSY054_SpMod_Day2\TSeries-10312025-1751-001\VRlog_JSY038_10312025_06-08-26_forSharing_forSharing.txt


## 2. Parse the raw log

The first 3 lines of the file are metadata (session start, log-format description, event-type
legend), not data -- same as `lines[3:]` in `helper/loadData.py`.

Column meaning depends on `EventType`:
- `p` (position) rows: `CurrentTime, ElapsedTime(seconds), EventType, Location(au)` -- 4 fields
- everything else (`s, n, z, r, t, f, e, sequence`): `CurrentTime, ElapsedTime(seconds), EventType,
  trial#, RewardLocation` -- up to 5 fields (for `sequence` rows, the last field is actually the
  shuffled reward-location order list, e.g. `[6,1,2,3,4,5]`)

Rows are padded to a common width so they load into one table without errors.

In [46]:

def load_vrlog_raw(path, tab=TAB):
    with open(path, "r") as f:
        lines = f.readlines()

    header_lines = lines[2]
    data_lines = lines[3:]

    rows = []
    for line in data_lines:
        line = line.strip()
        if not line:
            continue
        fields = line.split(tab)
        fields = fields + [np.nan] * (5 - len(fields))  # pad to 5 columns
        rows.append(fields[:5])

    df = pd.DataFrame(rows, columns=[
        "CurrentTime", "ElapsedTime(seconds)", "EventType", "trial_num", "reward_location"
    ])
    df["ElapsedTime(seconds)"] = pd.to_numeric(df["ElapsedTime(seconds)"], errors="coerce")
    return df, header_lines

vr_df, header_lines = load_vrlog_raw(VRLOG_PATH)
vr_df.head(15)

,CurrentTime,ElapsedTime(seconds),EventType,trial_num,reward_location
0,06.08.26.602116,0.00000,n,,
1,06.08.26.605108,0.00000,p,14.13300,4.84943
2,06.08.26.624153,0.00000,s,,
3,06.08.27.709740,0.35333,p,14.13300,4.84943
4,06.08.27.746613,0.42045,p,18.20004,6.24495
5,06.08.27.800927,0.47682,p,22.04156,7.56308
6,06.08.27.879716,0.55468,p,28.18011,9.66939
7,06.08.27.944546,0.61883,p,33.44376,11.47550
8,06.08.28.017500,0.69165,p,38.99582,13.38057
9,06.08.28.074347,0.74930,p,43.14847,14.80546


In [47]:
print(vr_df.head(3))

       CurrentTime  ElapsedTime(seconds) EventType trial_num reward_location
0  06.08.26.602116                   0.0         n                          
1  06.08.26.605108                   0.0         p  14.13300         4.84943
2  06.08.26.624153                   0.0         s                          


## 3. Position calibration (au -> cm)

Real logged position values run from ~0 up to ~393 AU per lap (confirmed from an actual
session), matching the fixed 393-AU cap used in `BehavioralDataFiltering.py` -- not the
~1320.6 AU implied by `single_lap_VR` in `Preprocess.py`'s spatial-discretization block. So
the conversion factor is computed the same way `process_data_with_speed_filtering` does: from
this log's own observed AU range, capped at 393 AU, scaled onto the physical lap length
(`single_lap_treadmill`, ~130.03 cm) computed from `Preprocess.py`'s wheel-calibration
constants.

In [48]:
single_revolution_VR = 282.415          # VR position units traveled per one wheel revolution
single_revolution_treadmill = 27.8      # physical treadmill distance (cm) per one wheel revolution
single_lap_VR = 1320.645683             # VR position units in one full lap
single_lap_treadmill = single_revolution_treadmill * single_lap_VR / single_revolution_VR

is_position_row = vr_df["EventType"] == "p"
pos_au = pd.to_numeric(vr_df.loc[is_position_row, "trial_num"], errors="coerce")

au_min, au_max = pos_au.min(), pos_au.max()
track_length_au = min(au_max, 393.0)          # matches BehavioralDataFiltering.py's fixed track-length cap
location_range_au = track_length_au - au_min
cm_per_au = single_lap_treadmill / location_range_au

print(f"Observed Location_au range: {au_min:.2f} to {au_max:.2f} AU")
print(f"Track length used (capped): {track_length_au:.2f} AU")
print(f"Target physical lap length (single_lap_treadmill): {single_lap_treadmill:.2f} cm")
print(f"Conversion factor: {cm_per_au:.6f} cm per AU")
print(f"Resulting Location_cm range: {au_min * cm_per_au:.2f} to {au_max * cm_per_au:.2f} cm")

Observed Location_au range: 14.13 to 585.31 AU
Track length used (capped): 393.00 AU
Target physical lap length (single_lap_treadmill): 130.00 cm
Conversion factor: 0.343128 cm per AU
Resulting Location_cm range: 4.85 to 200.84 cm


## 4. Add `Location_au` / `Location_cm` next to the position rows

Only `p` (position) rows carry an actual spatial sample -- `trial_num` on the other event
rows is a trial number, not a position, so `Location_au`/`Location_cm` are left blank there.

In [49]:
vr_df["Location_au"] = pd.to_numeric(vr_df["trial_num"], errors="coerce").where(is_position_row)
vr_df["Location_cm"] = vr_df["Location_au"] * cm_per_au

# For 'p' rows, the position value was sitting in the trial_num/reward_location slots
# (there's no real trial#/RewardLocation logged on those rows) -- now that it's been
# promoted to Location_au/Location_cm, blank those two out to avoid double-meaning columns.
vr_df.loc[is_position_row, ["trial_num", "reward_location"]] = np.nan

vr_df = vr_df[["CurrentTime", "ElapsedTime(seconds)", "EventType",
               "Location_au", "Location_cm", "trial_num", "reward_location"]]

print(f"Position ('p') rows: {int(is_position_row.sum())} / {len(vr_df)}")
vr_df.head(15)

Position ('p') rows: 38035 / 38271


,CurrentTime,ElapsedTime(seconds),EventType,Location_au,Location_cm,trial_num,reward_location
0,06.08.26.602116,0.00000,n,NaN,NaN,,
1,06.08.26.605108,0.00000,p,14.13300,4.849433,NaN,NaN
2,06.08.26.624153,0.00000,s,NaN,NaN,,
3,06.08.27.709740,0.35333,p,14.13300,4.849433,NaN,NaN
4,06.08.27.746613,0.42045,p,18.20004,6.244949,NaN,NaN
5,06.08.27.800927,0.47682,p,22.04156,7.563084,NaN,NaN
6,06.08.27.879716,0.55468,p,28.18011,9.669394,NaN,NaN
7,06.08.27.944546,0.61883,p,33.44376,11.475501,NaN,NaN
8,06.08.28.017500,0.69165,p,38.99582,13.380570,NaN,NaN
9,06.08.28.074347,0.74930,p,43.14847,14.805462,NaN,NaN


## 5. Remove duplicate rows (same `ElapsedTime(seconds)`)

Scoped to `p` (position) rows only: if two position rows share the same elapsed-time
value, keep only the first and drop the rest of that row entirely. `s`/`n`/`z`/`r`/`t`/`f`/
`sequence` marker rows are never dropped by this step, even if they happen to share a
timestamp with a position row or with each other (e.g. `z` and `r` commonly fire at the
same instant, and several markers can legitimately fire at ElapsedTime = 0 during session
start).

In [50]:
n_before = len(vr_df)

# Only check for duplicate ElapsedTime values within the position rows -- that is the
# only case where a repeated timestamp is actually a duplicate sample, not two distinct
# events that happen to fire in the same instant.
position_rows = vr_df.loc[is_position_row]
dup_within_position = position_rows.duplicated(subset="ElapsedTime(seconds)", keep="first")

duplicate_mask = pd.Series(False, index=vr_df.index)
duplicate_mask.loc[position_rows.index] = dup_within_position.values

print(f"Dropping {int(duplicate_mask.sum())} duplicate-timestamp position rows out of {n_before}")

# Sanity check: peek at a few of the dropped rows before committing
vr_df.loc[duplicate_mask].head(10)

Dropping 0 duplicate-timestamp position rows out of 38271


,CurrentTime,ElapsedTime(seconds),EventType,Location_au,Location_cm,trial_num,reward_location


In [51]:
vr_df_clean = vr_df.loc[~duplicate_mask].reset_index(drop=True)
print(f"Remaining rows: {len(vr_df_clean)}")
vr_df_clean.head(15)

Remaining rows: 38271


,CurrentTime,ElapsedTime(seconds),EventType,Location_au,Location_cm,trial_num,reward_location
0,06.08.26.602116,0.00000,n,NaN,NaN,,
1,06.08.26.605108,0.00000,p,14.13300,4.849433,NaN,NaN
2,06.08.26.624153,0.00000,s,NaN,NaN,,
3,06.08.27.709740,0.35333,p,14.13300,4.849433,NaN,NaN
4,06.08.27.746613,0.42045,p,18.20004,6.244949,NaN,NaN
5,06.08.27.800927,0.47682,p,22.04156,7.563084,NaN,NaN
6,06.08.27.879716,0.55468,p,28.18011,9.669394,NaN,NaN
7,06.08.27.944546,0.61883,p,33.44376,11.475501,NaN,NaN
8,06.08.28.017500,0.69165,p,38.99582,13.380570,NaN,NaN
9,06.08.28.074347,0.74930,p,43.14847,14.805462,NaN,NaN


## 6. Save for sharing (tab-delimited .txt)

In [52]:
vr_df_clean.to_csv(OUTPUT_PATH, sep=TAB, index=False, float_format="%.5f")
print("Saved cleaned log to:")
print(f"  {OUTPUT_PATH}")

Saved cleaned log to:
  D:\V1_SpatialModulation\2p\V1_prism\JSY054_ChronicImaging\251031_JSY_JSY054_SpMod_Day2\TSeries-10312025-1751-001\VRlog_JSY038_10312025_06-08-26_forSharing_forSharing.txt
